In [4]:
# Generic experiment-directory history ranking
#
# Paste this complete file into one notebook code cell, or run it with:
#
#     %run /path/to/generic_history_ranking_with_beta.py
#
# Change only EXPERIMENT_NAME:
#
#     "initial_test"
#     "continuous_test"
#     "modern_tcn_graph_sweep"
#
# The cell:
#   1. finds every history.csv recursively;
#   2. explicitly excludes every _sweep_control directory;
#   3. detects both the old slash-style and newer underscore-style
#      cumulative-log-change-MAE history columns;
#   4. recomputes the five-horizon mean at every epoch;
#   5. ranks runs by the lowest retrospective five-horizon mean;
#   6. displays one-step runs separately;
#   7. for modern_tcn_sweep, reports the learned spatial beta at the
#      same retrospective best epoch used for the ranking.
#
# The beta column is therefore internally aligned with the displayed
# Log-MAE values. No-graph runs display an em dash.

from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------------
# User settings.
# ------------------------------------------------------------------

EXPERIMENT_NAME = "modern_tcn_graph_sweep"

# Set this to an explicit directory to bypass automatic root discovery.
# Example:
#
# EXPERIMENT_DIR = Path(
#     "/Users/vishalruparelia/Library/CloudStorage/"
#     "GoogleDrive-vishal@autonomous-fox.ai/"
#     "My Drive/dissertation/final_model/modern_tcn_sweep"
# )
#
EXPERIMENT_DIR = None

# Set False to display only runs whose run_metadata.json says completed.
INCLUDE_INCOMPLETE_RUNS = True

EXPECTED_HORIZONS = (
    1,
    5,
    15,
    30,
    60,
)

SHOW_SPATIAL_BETA_AT_BEST_EPOCH = (
    EXPERIMENT_NAME.startswith("modern_tcn")
)


# ------------------------------------------------------------------
# Locate the requested experiment directory.
# ------------------------------------------------------------------

FINAL_MODEL_ROOT_CANDIDATES = (
    Path(
        "/content/drive/MyDrive/"
        "dissertation/final_model"
    ),
    Path(
        "/Users/vishalruparelia/Library/CloudStorage/"
        "GoogleDrive-vishal@autonomous-fox.ai/"
        "My Drive/dissertation/final_model"
    ),
)

if EXPERIMENT_DIR is not None:
    ROOT_DIR = Path(
        EXPERIMENT_DIR
    ).expanduser()
else:
    final_model_root = next(
        (
            path
            for path in FINAL_MODEL_ROOT_CANDIDATES
            if path.is_dir()
        ),
        None,
    )

    if final_model_root is None:
        raise FileNotFoundError(
            "Could not locate the dissertation final_model directory.\n"
            "Checked:\n"
            + "\n".join(
                f"  - {path}"
                for path in FINAL_MODEL_ROOT_CANDIDATES
            )
        )

    ROOT_DIR = (
        final_model_root
        / EXPERIMENT_NAME
    )

if not ROOT_DIR.is_dir():
    parent = ROOT_DIR.parent

    available_directories = (
        sorted(
            path.name
            for path in parent.iterdir()
            if path.is_dir()
        )
        if parent.is_dir()
        else []
    )

    raise FileNotFoundError(
        f"Experiment directory does not exist: {ROOT_DIR}\n\n"
        "Available sibling directories:\n"
        + (
            "\n".join(
                f"  - {name}"
                for name in available_directories
            )
            if available_directories
            else "  <none found>"
        )
    )

print("Scanning:")
print(ROOT_DIR)

if SHOW_SPATIAL_BETA_AT_BEST_EPOCH:
    print(
        "ModernTCN mode: the table will include `spatial_beta` from "
        "the same retrospective best epoch used for ranking."
    )


# ------------------------------------------------------------------
# Helpers.
# ------------------------------------------------------------------

def load_json_optional(
    path: Path,
) -> dict:
    """Load a JSON object when available, otherwise return {}."""

    if not path.is_file():
        return {}

    with path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def normalise_column_name(
    value: str,
) -> str:
    """
    Normalise slash-style, underscore-style and other separators.

    Examples:
        val/cumulative_log_change_mae/h1
        val_cumulative_log_change_mae_h1

    both become:
        val_cumulative_log_change_mae_h1
    """

    return (
        re.sub(
            r"[^a-z0-9]+",
            "_",
            str(value).strip().lower(),
        )
        .strip("_")
    )


def metric_column_priority(
    column_name: str,
    horizon: int,
) -> tuple[int, int]:
    """Prefer the canonical project history-column formats."""

    normalised = normalise_column_name(
        column_name
    )

    canonical = {
        f"val_cumulative_log_change_mae_h{horizon}": 0,
        (
            "validation_cumulative_log_change_mae_"
            f"h{horizon}"
        ): 1,
    }

    return (
        canonical.get(
            normalised,
            10,
        ),
        len(normalised),
    )


def detect_horizon_columns(
    columns,
) -> dict[int, str]:
    """
    Detect validation cumulative-log-change MAE columns.

    Supported examples include:
        val/cumulative_log_change_mae/h1
        val_cumulative_log_change_mae_h1
        validation_cumulative_log_change_mae_h1
    """

    candidates: dict[
        int,
        list[tuple[tuple[int, int], str]],
    ] = {}

    for column in columns:
        normalised = normalise_column_name(
            column
        )

        if (
            "cumulative_log_change_mae"
            not in normalised
        ):
            continue

        if not (
            normalised.startswith("val_")
            or normalised.startswith("validation_")
        ):
            continue

        horizon_match = re.search(
            r"(?:^|_)h(?:orizon)?_?(\d+)(?:_|$)",
            normalised,
        )

        if horizon_match is None:
            minute_match = re.search(
                r"(?:^|_)(\d+)_?min(?:_|$)",
                normalised,
            )

            if minute_match is None:
                continue

            horizon = int(
                minute_match.group(1)
            )
        else:
            horizon = int(
                horizon_match.group(1)
            )

        candidates.setdefault(
            horizon,
            [],
        ).append(
            (
                metric_column_priority(
                    column,
                    horizon,
                ),
                str(column),
            )
        )

    detected = {}

    for horizon, horizon_candidates in candidates.items():
        horizon_candidates.sort(
            key=lambda item: item[0]
        )

        detected[horizon] = (
            horizon_candidates[0][1]
        )

    return detected


def detect_spatial_beta_column(
    columns,
) -> str | None:
    """
    Locate the learned spatial-gate beta column.

    The current continuous/ModernTCN runner writes `spatial_beta`.
    The fallback suffix matching makes the analysis tolerant of an
    optional train/validation prefix in a later runner version.
    """

    exact_matches = []
    suffix_matches = []

    for column in columns:
        normalised = normalise_column_name(
            column
        )

        if normalised == "spatial_beta":
            exact_matches.append(
                str(column)
            )
        elif normalised.endswith(
            "_spatial_beta"
        ):
            suffix_matches.append(
                str(column)
            )

    if exact_matches:
        return sorted(
            exact_matches
        )[0]

    if suffix_matches:
        return sorted(
            suffix_matches,
            key=lambda value: (
                len(
                    normalise_column_name(
                        value
                    )
                ),
                value,
            ),
        )[0]

    return None


def detect_epoch_column(
    history: pd.DataFrame,
) -> str | None:
    """Find the epoch column used by a history table."""

    for candidate in (
        "epoch",
        "Epoch",
    ):
        if candidate in history.columns:
            return candidate

    return None


def epoch_from_row(
    history: pd.DataFrame,
    row_index,
) -> int:
    """Return the saved epoch value, falling back to the row index."""

    epoch_column = detect_epoch_column(
        history
    )

    if epoch_column is not None:
        value = history.loc[
            row_index,
            epoch_column,
        ]

        if pd.notna(value):
            return int(
                value
            )

    return int(
        row_index
    )


def spatial_beta_at_row(
    history: pd.DataFrame,
    row_index,
) -> float | None:
    """
    Return the learned spatial beta recorded on one history row.

    For graph-free runs the beta column may be absent or NaN; in that
    case None is returned.
    """

    beta_column = detect_spatial_beta_column(
        history.columns
    )

    if beta_column is None:
        return None

    value = pd.to_numeric(
        pd.Series(
            [history.loc[row_index, beta_column]]
        ),
        errors="coerce",
    ).iloc[0]

    if not np.isfinite(value):
        return None

    return float(value)


def beta_field_for_row(
    history: pd.DataFrame,
    row_index,
) -> dict:
    """Add beta from the same row used for the reported score."""

    if not SHOW_SPATIAL_BETA_AT_BEST_EPOCH:
        return {}

    beta = spatial_beta_at_row(
        history,
        row_index,
    )

    return {
        "Learned beta at best epoch": (
            np.nan
            if beta is None
            else beta
        ),
    }

def safe_saved_epoch(
    metadata: dict,
):
    """Read the selected checkpoint epoch when present."""

    value = metadata.get(
        "best_epoch"
    )

    if value is None:
        return np.nan

    try:
        return int(
            value
        )
    except (
        TypeError,
        ValueError,
    ):
        return np.nan


def relative_run_name(
    run_dir: Path,
) -> str:
    """Return a stable run identifier relative to the scanned root."""

    relative = run_dir.relative_to(
        ROOT_DIR
    )

    if str(
        relative
    ) == ".":
        return run_dir.name

    return relative.as_posix()




# ------------------------------------------------------------------
# Scan histories.
# ------------------------------------------------------------------

history_paths = sorted(
    path
    for path in ROOT_DIR.rglob(
        "history.csv"
    )
    if "_sweep_control" not in path.parts
)

if not history_paths:
    raise RuntimeError(
        f"No history.csv files were found under {ROOT_DIR}."
    )

five_horizon_rows = []
one_step_rows = []
partial_contract_rows = []
aggregate_only_rows = []
skipped_rows = []


for history_path in history_paths:
    run_dir = history_path.parent
    run_name = relative_run_name(
        run_dir
    )

    metadata = load_json_optional(
        run_dir
        / "run_metadata.json"
    )

    status = metadata.get(
        "status",
        "unknown",
    )

    if (
        not INCLUDE_INCOMPLETE_RUNS
        and status != "completed"
    ):
        skipped_rows.append(
            {
                "Run": run_name,
                "Reason": (
                    f"Status is {status!r}; "
                    "incomplete runs excluded"
                ),
            }
        )
        continue

    try:
        history = pd.read_csv(
            history_path
        )
    except Exception as error:
        skipped_rows.append(
            {
                "Run": run_name,
                "Reason": (
                    f"Could not read history.csv: {error}"
                ),
            }
        )
        continue

    if history.empty:
        skipped_rows.append(
            {
                "Run": run_name,
                "Reason": "history.csv is empty",
            }
        )
        continue

    horizon_columns = detect_horizon_columns(
        history.columns
    )

    available_horizons = tuple(
        sorted(
            horizon_columns
        )
    )

    saved_epoch = safe_saved_epoch(
        metadata
    )

    common = {
        "Run": run_name,
        "Status": status,
        "Epochs recorded": int(
            len(
                history
            )
        ),
        "Saved checkpoint epoch": saved_epoch,
    }

    # --------------------------------------------------------------
    # Standard five-horizon runs.
    # --------------------------------------------------------------

    if set(
        EXPECTED_HORIZONS
    ).issubset(
        horizon_columns
    ):
        ordered_columns = [
            horizon_columns[
                horizon
            ]
            for horizon in EXPECTED_HORIZONS
        ]

        metric_frame = (
            history[
                ordered_columns
            ]
            .apply(
                pd.to_numeric,
                errors="coerce",
            )
        )

        metric_frame.columns = list(
            EXPECTED_HORIZONS
        )

        # Require all five horizons to be finite on the same epoch.
        epoch_average = metric_frame.mean(
            axis=1,
            skipna=False,
        )

        finite_mask = np.isfinite(
            epoch_average.to_numpy(
                dtype=float
            )
        )

        valid_indices = epoch_average.index[
            finite_mask
        ]

        if len(
            valid_indices
        ) == 0:
            skipped_rows.append(
                {
                    "Run": run_name,
                    "Reason": (
                        "No epoch contains finite values for "
                        "all five horizons"
                    ),
                }
            )
            continue

        best_index = epoch_average.loc[
            valid_indices
        ].idxmin()

        best_epoch = epoch_from_row(
            history,
            best_index,
        )

        row = {
            **common,
            **beta_field_for_row(
                history,
                best_index,
            ),
            "Retrospective best epoch": best_epoch,
            "Matches saved epoch": (
                np.nan
                if pd.isna(
                    saved_epoch
                )
                else best_epoch == int(
                    saved_epoch
                )
            ),
            "Valid five-horizon epochs": int(
                len(
                    valid_indices
                )
            ),
            "Best average Log MAE": float(
                epoch_average.loc[
                    best_index
                ]
            ),
        }

        for horizon in EXPECTED_HORIZONS:
            row[
                f"Log MAE — {horizon} min"
            ] = float(
                metric_frame.loc[
                    best_index,
                    horizon,
                ]
            )

        five_horizon_rows.append(
            row
        )
        continue

    # --------------------------------------------------------------
    # One-step runs.
    # --------------------------------------------------------------

    if available_horizons == (
        1,
    ):
        metric_series = pd.to_numeric(
            history[
                horizon_columns[
                    1
                ]
            ],
            errors="coerce",
        )

        valid = metric_series.loc[
            np.isfinite(
                metric_series.to_numpy(
                    dtype=float
                )
            )
        ]

        if valid.empty:
            skipped_rows.append(
                {
                    "Run": run_name,
                    "Reason": (
                        "No finite one-minute Log MAE values"
                    ),
                }
            )
            continue

        best_index = valid.idxmin()
        best_epoch = epoch_from_row(
            history,
            best_index,
        )

        one_step_rows.append(
            {
                **common,
                **beta_field_for_row(
                    history,
                    best_index,
                ),
                "Retrospective best epoch": best_epoch,
                "Matches saved epoch": (
                    np.nan
                    if pd.isna(
                        saved_epoch
                    )
                    else best_epoch == int(
                        saved_epoch
                    )
                ),
                "Best Log MAE — 1 min": float(
                    valid.loc[
                        best_index
                    ]
                ),
            }
        )
        continue

    # --------------------------------------------------------------
    # Other partial horizon contracts.
    # --------------------------------------------------------------

    if available_horizons:
        partial_contract_rows.append(
            {
                **common,
                "Available horizons": ", ".join(
                    str(
                        horizon
                    )
                    for horizon in available_horizons
                ),
                "Reason": (
                    "Not the complete [1, 5, 15, 30, 60] contract"
                ),
            }
        )
        continue

    # --------------------------------------------------------------
    # Legacy aggregate-only histories.
    # --------------------------------------------------------------

    if "all_horizons_score" in history.columns:
        aggregate = pd.to_numeric(
            history[
                "all_horizons_score"
            ],
            errors="coerce",
        )

        valid = aggregate.loc[
            np.isfinite(
                aggregate.to_numpy(
                    dtype=float
                )
            )
        ]

        if not valid.empty:
            best_index = valid.idxmin()

            aggregate_only_rows.append(
                {
                    **common,
                    **beta_field_for_row(
                        history,
                        best_index,
                    ),
                    "Retrospective best epoch": (
                        epoch_from_row(
                            history,
                            best_index,
                        )
                    ),
                    "Best aggregate score": float(
                        valid.loc[
                            best_index
                        ]
                    ),
                    "Warning": (
                        "No per-horizon columns; "
                        "horizon contract not verified"
                    ),
                }
            )
            continue

    skipped_rows.append(
        {
            "Run": run_name,
            "Reason": (
                "No recognised validation cumulative-log-change "
                "MAE columns"
            ),
        }
    )


# ------------------------------------------------------------------
# Display helpers.
# ------------------------------------------------------------------

def display_format_map() -> dict:
    """Build a format mapping appropriate to the active experiment."""

    format_map = {
        "Saved checkpoint epoch": "{:.0f}",
        "Retrospective best epoch": "{:.0f}",
        "Best average Log MAE": "{:.8f}",
        "Best Log MAE — 1 min": "{:.8f}",
        "Best aggregate score": "{:.8f}",
        "Log MAE — 1 min": "{:.8f}",
        "Log MAE — 5 min": "{:.8f}",
        "Log MAE — 15 min": "{:.8f}",
        "Log MAE — 30 min": "{:.8f}",
        "Log MAE — 60 min": "{:.8f}",
    }

    if SHOW_SPATIAL_BETA_AT_BEST_EPOCH:
        format_map[
            "Learned beta at best epoch"
        ] = "{:.4f}"

    return format_map


FORMAT_MAP = display_format_map()


# ------------------------------------------------------------------
# Display five-horizon ranking.
# ------------------------------------------------------------------

print()
print(
    f"Found {len(history_paths)} history.csv files "
    "after excluding _sweep_control."
)

if five_horizon_rows:
    five_horizon_results = (
        pd.DataFrame(
            five_horizon_rows
        )
        .sort_values(
            "Best average Log MAE",
            ascending=True,
        )
        .reset_index(
            drop=True
        )
    )

    five_horizon_results.insert(
        0,
        "Rank",
        np.arange(
            1,
            len(
                five_horizon_results
            ) + 1,
        ),
    )

    display(
        five_horizon_results.style
        .format(
            FORMAT_MAP,
            na_rep="—",
        )
        .highlight_min(
            subset=[
                "Best average Log MAE",
            ],
            axis=0,
        )
        .set_caption(
            f"{EXPERIMENT_NAME}: five-horizon runs ranked by "
            "the lowest retrospective mean Log MAE in history.csv"
        )
    )

    winner = five_horizon_results.iloc[
        0
    ]

    print()
    print("Best five-horizon run:")
    print(
        "  Run:",
        winner[
            "Run"
        ],
    )
    print(
        "  Epoch:",
        int(
            winner[
                "Retrospective best epoch"
            ]
        ),
    )
    print(
        "  Average Log MAE:",
        f"{winner['Best average Log MAE']:.8f}",
    )

    if SHOW_SPATIAL_BETA_AT_BEST_EPOCH:
        winner_beta = winner.get(
            "Learned beta at best epoch"
        )

        print(
            "  Learned beta at best epoch:",
            (
                "—"
                if pd.isna(
                    winner_beta
                )
                else f"{float(winner_beta):.4f}"
            ),
        )
else:
    print(
        "No complete five-horizon run histories were found."
    )


# ------------------------------------------------------------------
# Display one-step runs separately.
# ------------------------------------------------------------------

if one_step_rows:
    one_step_results = (
        pd.DataFrame(
            one_step_rows
        )
        .sort_values(
            "Best Log MAE — 1 min",
            ascending=True,
        )
        .reset_index(
            drop=True
        )
    )

    one_step_results.insert(
        0,
        "Rank",
        np.arange(
            1,
            len(
                one_step_results
            ) + 1,
        ),
    )

    display(
        one_step_results.style
        .format(
            FORMAT_MAP,
            na_rep="—",
        )
        .highlight_min(
            subset=[
                "Best Log MAE — 1 min",
            ],
            axis=0,
        )
        .set_caption(
            f"{EXPERIMENT_NAME}: one-step runs ranked separately"
        )
    )


# ------------------------------------------------------------------
# Display histories that cannot be mixed into the main ranking.
# ------------------------------------------------------------------

if partial_contract_rows:
    print()
    print(
        "Runs with partial/non-standard horizon contracts:"
    )

    display(
        pd.DataFrame(
            partial_contract_rows
        )
        .style
        .format(
            FORMAT_MAP,
            na_rep="—",
        )
    )


if aggregate_only_rows:
    print()
    print(
        "Legacy aggregate-only histories "
        "(not mixed into the verified five-horizon ranking):"
    )

    display(
        pd.DataFrame(
            aggregate_only_rows
        )
        .sort_values(
            "Best aggregate score"
        )
        .style
        .format(
            FORMAT_MAP,
            na_rep="—",
        )
    )


if skipped_rows:
    print()
    print("Skipped histories:")

    display(
        pd.DataFrame(
            skipped_rows
        )
    )


Scanning:
/Users/vishalruparelia/Library/CloudStorage/GoogleDrive-vishal@autonomous-fox.ai/My Drive/dissertation/final_model/modern_tcn_graph_sweep
ModernTCN mode: the table will include `spatial_beta` from the same retrospective best epoch used for ranking.

Found 68 history.csv files after excluding _sweep_control.


,Rank,Run,Status,Epochs recorded,Saved checkpoint epoch,Learned beta at best epoch,Retrospective best epoch,Matches saved epoch,Valid five-horizon epochs,Best average Log MAE,Log MAE — 1 min,Log MAE — 5 min,Log MAE — 15 min,Log MAE — 30 min,Log MAE — 60 min
0,1,mtg_beta_d32_k1_p8s4_lk15_dynamic_g1_h32_lr0p0001_glr0p0005_b0p5_fp32gate_v1,completed,43,33,0.4790,33,True,43,0.00148252,0.00038882,0.00081424,0.00145169,0.00198632,0.00277152
1,2,mtg_s3_d32_k1_p8s4_lk15_dynamic_g1_h32_lr0p0001_glr0p0005,completed,43,33,0.4790,33,True,43,0.00148252,0.00038882,0.00081424,0.00145169,0.00198632,0.00277152
2,3,mtg_beta_d32_k1_p8s4_lk15_dynamic_g1_h32_lr0p0001_glr0p0005_b0p3_fp32gate_v1,completed,43,33,0.2881,33,True,43,0.00148258,0.00038783,0.00081416,0.00145241,0.00198652,0.00277195
3,4,mtg_s3_d32_k1_p8s4_lk15_dynamic_g1_h32_lr0p00025_glr0p0005,completed,43,33,0.4744,33,True,43,0.00148274,0.00038736,0.00081314,0.00145262,0.00198660,0.00277397
4,5,mtg_s2_d32_k1_p8s4_lk15_baseglr0p001_dynamic_g1_h32,completed,43,33,0.4797,33,True,43,0.00148275,0.00038879,0.00081430,0.00145192,0.00198679,0.00277195
5,6,mtg_s3_d32_k1_p4s2_lk51_free_static_g2_lr0p0001_glr0p002,completed,43,33,0.4800,33,True,43,0.00148278,0.00039163,0.00081190,0.00145083,0.00198633,0.00277321
6,7,mtg_beta_d32_k1_p8s4_lk15_dynamic_g1_h32_lr0p0001_glr0p0005_b0p1_fp32gate_v1,completed,43,33,0.0997,33,True,43,0.00148279,0.00038753,0.00081379,0.00145272,0.00198684,0.00277305
7,8,mtg_s3_d32_k1_p8s4_lk15_dynamic_g1_h32_lr0p00025_glr0p001,completed,43,33,0.4756,33,True,43,0.00148281,0.00038730,0.00081320,0.00145264,0.00198681,0.00277411
8,9,mtg_s3_d32_k1_p8s4_lk15_dynamic_g1_h32_lr0p00025_glr0p002,completed,43,33,0.4763,33,True,43,0.00148301,0.00038733,0.00081352,0.00145282,0.00198710,0.00277430
9,10,mtg_s3_d32_k1_p4s2_lk51_free_static_g2_lr0p00025_glr0p002,completed,43,33,0.4773,33,True,43,0.00148303,0.00038964,0.00081178,0.00145307,0.00198708,0.00277359



Best five-horizon run:
  Run: mtg_beta_d32_k1_p8s4_lk15_dynamic_g1_h32_lr0p0001_glr0p0005_b0p5_fp32gate_v1
  Epoch: 33
  Average Log MAE: 0.00148252
  Learned beta at best epoch: 0.4790
